# Member-4: Digital Twin + Counterfactual Simulator + RL Engine (DT-RAM)

**Scope (CODEOWNERS):** `services/rl-service/`, `services/simulation-service/`, shared `services/safety-service/`.
Python proposes decisions only — the Go infrastructure-controller (Member-2) owns Kubernetes actuation.
Single-agent PPO baseline first; simulation is the training environment.

Sections: 1) Setup · 2) Digital Twin · 3) Counterfactual simulation · 4) PPO baseline ·
5) Safety gate · 6) Decision emit (§54) · 7) Reproducibility footer.

In [ ]:
import os, sys

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
REPO_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", "..", ".."))
sys.path.insert(0, os.path.join(REPO_ROOT, "services", "rl-service"))
sys.path.insert(0, os.path.join(REPO_ROOT, "services", "simulation-service"))

SEED = 42
CONFIG = {
    "system": {"decision_interval_seconds": 5},
    "digital_twin": {"sync_interval_seconds": 1, "fidelity_threshold": 0.90, "autonomous_threshold": 0.95},
    "rl": {"algorithm": "ppo", "gamma": 0.99, "learning_rate": 0.0003},
    "safety": {"max_migrations_per_minute": 3, "minimum_prediction_confidence": 0.80, "max_slo_probability": 0.10},
}
print("repo:", REPO_ROOT, "| seed:", SEED)

## 2) Digital Twin — synchronized state, staleness, fidelity (§§16–18)
Build the twin from synthetic telemetry, flag stale nodes, and score fidelity.

In [ ]:
from datetime import datetime, timedelta, timezone
from app.twin import TwinState, WorkloadState, autonomy_level

twin = TwinState(stale_after_seconds=5.0)
twin.apply_telemetry("edge-a1", cpu_util=0.70, mem_util=0.55, network_in_mbps=210.0, network_out_mbps=180.0)
twin.apply_telemetry("edge-b1", cpu_util=0.35, mem_util=0.40, network_in_mbps=90.0, network_out_mbps=70.0)
# One stale node (timestamp far in the past) to demo fail-closed behaviour.
twin.apply_telemetry("edge-c1", cpu_util=0.50, mem_util=0.50,
                     timestamp=datetime.now(timezone.utc) - timedelta(seconds=3600))
twin.upsert_workload(WorkloadState(service_id="processor", node_id="edge-a1", replicas=3,
                                   cpu_usage=0.72, request_rate=182.0, p95_latency_ms=34.0, slo_ms=50.0))

physical = {
    "edge-a1": {"cpu": 0.72, "memory": 0.56, "network_in": 212.0, "network_out": 181.0},
    "edge-b1": {"cpu": 0.34, "memory": 0.41, "network_in": 91.0, "network_out": 69.0},
    "edge-c1": {"cpu": 0.90, "memory": 0.90, "network_in": 400.0, "network_out": 400.0},
}
scores = twin.score_all(physical)
for n in twin.list_nodes():
    print(f"{n.node_id}: fidelity={n.fidelity:.4f} stale={n.stale} "
          f"autonomy={autonomy_level(n.fidelity, n.stale)}")

## 3) Counterfactual simulation — what if we act? (§§29–31)
Evaluate no-op vs migrate vs scale vs place against the twin. Migration is costed, never free.

In [ ]:
from app.simulator import evaluate_candidates

candidates = [
    {"type": "NOOP", "service": "processor", "source": "edge-a1"},
    {"type": "MIGRATE", "service": "processor", "source": "edge-a1", "destination": "edge-b1"},
    {"type": "MIGRATE", "service": "processor", "source": "edge-a1", "destination": "edge-c1"},
    {"type": "SCALE", "service": "processor", "source": "edge-a1", "replicas": 4},
    {"type": "PLACE", "service": "processor", "source": "edge-a1", "destination": "edge-b1"},
]
results = evaluate_candidates(twin, candidates, service_cpu=0.5, service_mem_mb=512.0)
print(f"{'action':28s} {'cpu':>6s} {'lat_ms':>7s} {'slo_p':>6s} {'mig_cost':>9s} {'feasible':>9s}")
for c, r in zip(candidates, results):
    label = f"{c['type']}->{c.get('destination', '-')}"
    print(f"{label:28s} {r.predicted_cpu:6.3f} {r.predicted_latency_ms:7.2f} "
          f"{r.slo_violation_prob:6.3f} {r.migration_cost:9.2f} {str(r.feasible):>9s}  {'; '.join(r.notes)}")

## 4) PPO baseline — single-agent MDP (§§22–27)
State = current + predicted horizon; discrete actions; SLO-dominant multi-objective reward.
Training runs against the simulator, never physical infrastructure.

In [ ]:
from rl_app.env import EdgeResourceEnv, EnvConfig
from rl_app.ppo import PPOConfig, make_model, rollout
from rl_app.policies import policy_record

env = EdgeResourceEnv(EnvConfig(num_nodes=3, horizon=4, seed=SEED))
print(env.describe())
obs, info = env.reset(seed=SEED)
print("obs dim:", obs.shape, "| actions:", env.action_space.n)

# Deterministic random-policy rollout (seeded) — smoke of MDP dynamics.
import numpy as np
rng = np.random.default_rng(SEED)
total = 0.0
for _ in range(16):
    obs, reward, terminated, truncated, info = env.step(int(rng.integers(env.action_space.n)))
    total += reward
    if terminated or truncated:
        break
print(f"random-policy rollout reward={total:.3f} last={info}")

In [ ]:
# PPO training baseline (SB3). Keep timesteps small here; full training
# belongs to experiment branches with provenance (Design §57), not CI.
ppo_cfg = PPOConfig(total_timesteps=2048, seed=SEED)
model = make_model(EnvConfig(num_nodes=3, horizon=4, seed=SEED), ppo_cfg)
model.learn(total_timesteps=ppo_cfg.total_timesteps)
eval_env = EdgeResourceEnv(EnvConfig(num_nodes=3, horizon=4, seed=SEED + 1))
print(rollout(model, eval_env, episodes=2))
print(policy_record("ppo-edge-migration", "v0.1.0", "synthetic-seed-42",
                    ppo_cfg.as_dict(), {"note": "baseline, simulation-only"}))

## 5) Safety gate — explicit checks, fail-closed (§§33–36, §64)
Every candidate passes capacity, SLO, health, fidelity, confidence, storm and cooldown gates.

In [ ]:
from app.safety import GateConfig, SafetyGate

gate = SafetyGate(GateConfig())
decisions = []
for c, r in zip(candidates, results):
    req = {
        "decision_id": f"nb-{c['type']}-{c.get('destination', 'na')}",
        "action": c,
        "predicted": {"slo_violation_probability": r.slo_violation_prob, "prediction_confidence": 0.87},
        "dt_fidelity": scores.get(c.get("source", ""), 0.0),
        "destination_free_cpu": 2.0,
        "destination_healthy": (c.get("destination", "edge-b1") != "edge-c1"),
        "source_healthy": True,
    }
    v = gate.evaluate(req)
    decisions.append((req["decision_id"], v))
    print(f"{req['decision_id']:28s} approved={v.approved!s:5s} reasons={v.reasons}")

## 6) Decision emit — propose, never actuate (§54, §66)
The notebook builds a `RESOURCE_DECISION` event. Execution belongs to the Go
infrastructure-controller after its own safety validation.

In [ ]:
import json
from app.schemas import resource_decision

approved = [(d, v) for d, v in decisions if v.approved]
if approved:
    did, verdict = approved[0]
    event = resource_decision(
        decision_id=did,
        policy_version="ppo-v0.1.0",
        action={"type": "MIGRATE", "service": "processor", "source": "edge-a1", "destination": "edge-b1"},
        predicted={"latency": 21.3, "cpu": 0.64, "slo_violation_probability": 0.04},
        approved=True,
    )
    print(json.dumps(event, indent=2))
else:
    print("No candidate approved — correct fail-closed outcome; emit nothing.")

## 7) Reproducibility footer (§§56–57)
Experiment progression: static → reactive → prediction → RL → DT+RL → DT+prediction+RL →
counterfactual DT-RL → full system + safety. This notebook covers DT + sim + PPO + safety.

In [ ]:
import json as _json
from rl_app.policies import git_sha

print(_json.dumps({"git_commit": git_sha(), "seed": SEED, "config": CONFIG},
                  indent=2, default=str))